# Lab | Langchain Evaluation

## Intro

Pick different sets of data and re-run this notebook. The point is for you to understand all steps involve and the many different ways one can and should evaluate LLM applications.

What did you learn? - Let's discuss that in class

## LangChain: Evaluation

### Outline:

* Example generation
* Manual evaluation (and debuging)
* LLM-assisted evaluation

In [1]:
!pip cache purge

Files removed: 0


In [2]:
# 1. Restart your kernel first
# 2. Use a relaxed version constraint to let pip resolve compatible sub-packages
!pip install langchain==0.2.16 langchain-community==0.2.16 langchain-openai langchain-huggingface docarray ragas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.3 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of langchain-huggingface to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain-huggingface to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of

### Example 1

#### Create our QandA application

In [1]:
from langchain.chains import RetrievalQA, LLMChain  # Not langchain_classic
from langchain.indexes import VectorstoreIndexCreator
from langchain_openai import ChatOpenAI, OpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import CSVLoader, TextLoader
from langchain_community.vectorstores import DocArrayInMemorySearch

In [2]:
file = '/content/data/OutdoorClothingCatalog_1000.csv'
loader = CSVLoader(file_path=file)
data = loader.load()

In [3]:
index = VectorstoreIndexCreator(
    vectorstore_cls=DocArrayInMemorySearch,
    embedding=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2", model_kwargs = {'device': 'cpu'})
).from_loaders([loader])




modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [4]:

vectorstore = index.vectorstore  # Get the vectorstore
retriever = vectorstore.as_retriever()  # Get the retriever

In [5]:
from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

In [6]:
llm = ChatOpenAI(temperature = 0.0, openai_api_key=OPENAI_API_KEY)
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=index.vectorstore.as_retriever(),
    verbose=True,
    chain_type_kwargs = {
        "document_separator": "<<<<>>>>>"
    }
)

#### Coming up with test datapoints

In [7]:
data[10]

Document(metadata={'source': '/content/data/OutdoorClothingCatalog_1000.csv', 'row': 10}, page_content=": 10\nname: Cozy Comfort Pullover Set, Stripe\ndescription: Perfect for lounging, this striped knit set lives up to its name. We used ultrasoft fabric and an easy design that's as comfortable at bedtime as it is when we have to make a quick run out.\r\n\r\nSize & Fit\r\n- Pants are Favorite Fit: Sits lower on the waist.\r\n- Relaxed Fit: Our most generous fit sits farthest from the body.\r\n\r\nFabric & Care\r\n- In the softest blend of 63% polyester, 35% rayon and 2% spandex.\r\n\r\nAdditional Features\r\n- Relaxed fit top with raglan sleeves and rounded hem.\r\n- Pull-on pants have a wide elastic waistband and drawstring, side pockets and a modern slim leg.\r\n\r\nImported.")

In [8]:
data[11]

Document(metadata={'source': '/content/data/OutdoorClothingCatalog_1000.csv', 'row': 11}, page_content=': 11\nname: Ultra-Lofty 850 Stretch Down Hooded Jacket\ndescription: This technical stretch down jacket from our DownTek collection is sure to keep you warm and comfortable with its full-stretch construction providing exceptional range of motion. With a slightly fitted style that falls at the hip and best with a midweight layer, this jacket is suitable for light activity up to 20° and moderate activity up to -30°. The soft and durable 100% polyester shell offers complete windproof protection and is insulated with warm, lofty goose down. Other features include welded baffles for a no-stitch construction and excellent stretch, an adjustable hood, an interior media port and mesh stash pocket and a hem drawcord. Machine wash and dry. Imported.')

#### Hard-coded examples

In [9]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import BaseOutputParser
from pydantic import BaseModel, Field

examples = [
    {
        "query": "Do the Cozy Comfort Pullover Set\
        have side pockets?",
        "answer": "Yes"
    },
    {
        "query": "What collection is the Ultra-Lofty \
        850 Stretch Down Hooded Jacket from?",
        "answer": "The DownTek collection"
    }
]

# Define the prompt template
prompt_template = PromptTemplate(
    input_variables=["query"],
    template="Examples:\n"
             "1. Query: Do the Cozy Comfort Pullover Set have side pockets?\n"
             "   Answer: Yes\n"
             "2. Query: What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?\n"
             "   Answer: The DownTek collection\n"
             "Query: {query}\n"
             "Answer:"
)

# Define the output model
class Answer(BaseModel):
    answer: str = Field(description="The answer to the query")

# Create the output parser
class AnswerOutputParser(BaseOutputParser):
    def parse(self, text: str) -> Answer:
        # Split the response to get the answer
        answer = text.strip().split("Answer:")[-1].strip()
        return Answer(answer=answer)

# Initialize the LLM
# llm = OpenAI()
llm = ChatOpenAI(openai_api_key=OPENAI_API_KEY)

# Create the LLMChain
llm_chain = LLMChain(
    llm=llm,
    prompt=prompt_template,
    output_parser=AnswerOutputParser()
)

# Example query
query = "Is the Cozy Comfort Pullover Set available in different colors?"

# Run the chain
result = llm_chain.run({"query": query})

# Print the result
print(result)


/tmp/ipykernel_3094/1022454896.py:46: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use RunnableSequence, e.g., `prompt | llm` instead.
  llm_chain = LLMChain(
/tmp/ipykernel_3094/1022454896.py:56: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use invoke instead.
  result = llm_chain.run({"query": query})


answer='Yes, it is available in four different colors: gray, navy, black, and cream.'


#### LLM-Generated examples

In [10]:
import langchain.evaluation.qa
print(dir(langchain.evaluation.qa))

['ContextQAEvalChain', 'CotQAEvalChain', 'QAEvalChain', 'QAGenerateChain', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'eval_chain', 'eval_prompt', 'generate_chain', 'generate_prompt']


In [11]:
from langchain.evaluation.qa import QAGenerateChain

In [12]:
# 1. Initialize the LLM
llm = ChatOpenAI(model="gpt-4o", openai_api_key=OPENAI_API_KEY) # Use your preferred model

# 2. Create the generation chain
example_gen_chain = QAGenerateChain.from_llm(llm)

In [13]:
llm_chain = LLMChain(llm=llm, prompt=prompt_template)

In [14]:
# FIX: apply_and_parse is deprecated and triggers a token-aggregation bug
# Process documents one at a time with invoke() to avoid both issues
new_examples = []
for doc in data[:5]:
    result_gen = example_gen_chain.invoke({'doc': doc})
    new_examples.append(result_gen)


In [15]:
# FIX: renamed loop var from 'data' (shadows the loaded dataset) to 'ex'
d_flattened = [ex['qa_pairs'] for ex in new_examples]
d_flattened


[{'query': "What are the key features of the Women's Campside Oxfords that contribute to their comfort and durability, as described in the document?",
  'answer': "The Women's Campside Oxfords are designed for comfort and durability with several key features. They have a super-soft canvas material for a broken-in feel and look, a comfortable EVA innersole with Cleansport NXT® antimicrobial odor control, and a moderate arch contour. Additionally, they include an EVA foam midsole for cushioning and support, and a chain-tread-inspired molded rubber outsole with a modified chain-tread pattern. These elements together ensure a comfortable and durable wear."},
 {'query': 'What materials are used in the construction of the Recycled Waterhog Dog Mat, and what are some of its key features?',
  'answer': 'The Recycled Waterhog Dog Mat is constructed with 24 oz. polyester fabric made from 94% recycled materials and a rubber backing. Key features of the mat include its ultradurable nature, the abi

#### Combine examples

In [16]:
examples

[{'query': 'Do the Cozy Comfort Pullover Set        have side pockets?',
  'answer': 'Yes'},
 {'query': 'What collection is the Ultra-Lofty         850 Stretch Down Hooded Jacket from?',
  'answer': 'The DownTek collection'}]

In [17]:
# examples += new_example
examples += d_flattened

In [18]:
examples[0]

{'query': 'Do the Cozy Comfort Pullover Set        have side pockets?',
 'answer': 'Yes'}

In [19]:
qa.invoke(examples[0]["query"])



> Entering new RetrievalQA chain...

> Finished chain.


{'query': 'Do the Cozy Comfort Pullover Set        have side pockets?',
 'result': 'Yes, the Cozy Comfort Pullover Set has side seam pockets and a back zip pocket, as well as two elastic mesh water bottle pockets.'}

### Manual Evaluation - Fun part

In [20]:
import langchain
langchain.debug = True

# qa needs 'query' as input, and it handles context internally
result = qa.invoke({
    "query": examples[0]["query"]
})
print(result)

[chain/start] [chain:RetrievalQA] Entering Chain run with input:
{
  "query": "Do the Cozy Comfort Pullover Set        have side pockets?"
}
[chain/start] [chain:RetrievalQA > chain:StuffDocumentsChain] Entering Chain run with input:
[inputs]
[chain/start] [chain:RetrievalQA > chain:StuffDocumentsChain > chain:LLMChain] Entering Chain run with input:
{
  "question": "Do the Cozy Comfort Pullover Set        have side pockets?",
  "context": "Side seam pockets and back zip pocket, with mesh insert for quick drainage.<<<<>>>>>Two elastic mesh water bottle pockets.\r\nTop compartment includes pocket with double-seal zipper for quick access.\r\nSide<<<<>>>>>All pockets have sturdy pocket bags and offer plenty of room for a wallet, cell phone and more.\r\n\r\nGusseted crotch for ease of movement.\r\n\r\nImported.<<<<>>>>>Two elastic mesh water bottle pockets.\r\nTop compartment includes pocket with double-se"
}
[llm/start] [chain:RetrievalQA > chain:StuffDocumentsChain > chain:LLMChain > llm

In [21]:
# Turn off the debug mode
langchain.debug = False

### LLM assisted evaluation

In [22]:
# When creating the RetrievalQA chain, add return_source_documents=True
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True  # Add this
)

In [23]:
predictions = []
for i, eg in enumerate(examples[:5]):
    result = qa.invoke({"query": eg["query"]})

    pred_dict = {
        'query': eg["query"],
        'answer': eg["answer"],
        'result': result['result'],
        'source_documents': result.get('source_documents', [])
    }
    predictions.append(pred_dict)

for i, eg in enumerate(examples[:5]):
    print(f"Example {i}:")
    print("Question:", predictions[i]['query'])
    print("Real Answer:", predictions[i]['answer'])
    print("Predicted Answer:", predictions[i]['result'])
    print(f"Used {len(predictions[i]['source_documents'])} source documents")
    print()

Example 0:
Question: Do the Cozy Comfort Pullover Set        have side pockets?
Real Answer: Yes
Predicted Answer: I don't know. The provided context does not mention the Cozy Comfort Pullover Set or any details about its features, including whether it has side pockets.
Used 4 source documents

Example 1:
Question: What collection is the Ultra-Lofty         850 Stretch Down Hooded Jacket from?
Real Answer: The DownTek collection
Predicted Answer: The Ultra-Lofty 850 Stretch Down Hooded Jacket is from the DownTek collection.
Used 4 source documents

Example 2:
Question: What are the key features of the Women's Campside Oxfords that contribute to their comfort and durability, as described in the document?
Real Answer: The Women's Campside Oxfords are designed for comfort and durability with several key features. They have a super-soft canvas material for a broken-in feel and look, a comfortable EVA innersole with Cleansport NXT® antimicrobial odor control, and a moderate arch contour. Ad

LLM as a JUdge

In [24]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser  # Add this import

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0,openai_api_key=OPENAI_API_KEY)

# Simple eval prompt (no classes)
eval_prompt = PromptTemplate(
    input_variables=["query", "answer", "result"],
    template="""Score PREDICTED vs ANSWER for QUERY (0-1, higher=better match).

Query: {query}
Answer: {answer}
Predicted: {result}

Score:"""
)

eval_chain = eval_prompt | llm | StrOutputParser()  # Now this will work

# Your EXACT workflow (no classes)
graded_outputs = []
for i, eg in enumerate(predictions):
    score_raw = eval_chain.invoke({
        "query": predictions[i]['query'],
        "answer": predictions[i]['answer'],
        "result": predictions[i]['result']
    })
    score = float(score_raw.strip())  # Extract number

    graded_outputs.append({'score': score})

    print(f"Example {i}:")
    print("Question:", predictions[i]['query'])
    print("Real:", predictions[i]['answer'])
    print("Predicted:", predictions[i]['result'])
    print("Score:", score, "\n")

Example 0:
Question: Do the Cozy Comfort Pullover Set        have side pockets?
Real: Yes
Predicted: I don't know. The provided context does not mention the Cozy Comfort Pullover Set or any details about its features, including whether it has side pockets.
Score: 0.5 

Example 1:
Question: What collection is the Ultra-Lofty         850 Stretch Down Hooded Jacket from?
Real: The DownTek collection
Predicted: The Ultra-Lofty 850 Stretch Down Hooded Jacket is from the DownTek collection.
Score: 1.0 

Example 2:
Question: What are the key features of the Women's Campside Oxfords that contribute to their comfort and durability, as described in the document?
Real: The Women's Campside Oxfords are designed for comfort and durability with several key features. They have a super-soft canvas material for a broken-in feel and look, a comfortable EVA innersole with Cleansport NXT® antimicrobial odor control, and a moderate arch contour. Additionally, they include an EVA foam midsole for cushioning

### Example 2
One can also easily evaluate your QA chains with the metrics offered in ragas

In [26]:
from langchain_huggingface import HuggingFaceEmbeddings
loader = TextLoader("/content/data/nyc_text.txt")
# FIX: changed device from 'cuda' to 'cpu' — use 'cuda' if you have a GPU
index = VectorstoreIndexCreator(
    embedding=HuggingFaceEmbeddings(
        model_name="all-MiniLM-L6-v2",
        model_kwargs={'device': 'cpu'}
    )
).from_loaders([loader])

retriever = index.vectorstore.as_retriever()

llm = ChatOpenAI(temperature=0, openai_api_key=OPENAI_API_KEY)
qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=retriever,
    return_source_documents=True,
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/langchain/indexes/vectorstore.py:127: UserWarning: Using InMemoryVectorStore as the default vectorstore.This memory store won't persist data. You should explicitlyspecify a vectorstore when using VectorstoreIndexCreator
  warnings.warn(


In [27]:
# testing it out

question = "How did New York City get its name?"
result = qa_chain.invoke({"query": question})
result["result"]

'New York City was originally named New Amsterdam by Dutch colonists in 1626. When the city came under British control in 1664, it was renamed New York after King Charles II of England granted the lands to his brother, the Duke of York. The city has been continuously named New York since November 1674.'

Now in order to evaluate the qa system we generated a few relevant questions. We've generated a few question for you but feel free to add any you want.

In [28]:
eval_questions = [
    "What is the population of New York City as of 2020?",
    "Which borough of New York City has the highest population?",
    "What is the economic significance of New York City?",
    "How did New York City get its name?",
    "What is the significance of the Statue of Liberty in New York City?",
]

eval_answers = [
    "8,804,190",
    "Brooklyn",
    "New York City's economic significance is vast, as it serves as the global financial capital, housing Wall Street and major financial institutions. Its diverse economy spans technology, media, healthcare, education, and more, making it resilient to economic fluctuations. NYC is a hub for international business, attracting global companies, and boasts a large, skilled labor force. Its real estate market, tourism, cultural industries, and educational institutions further fuel its economic prowess. The city's transportation network and global influence amplify its impact on the world stage, solidifying its status as a vital economic player and cultural epicenter.",
    "New York City got its name when it came under British control in 1664. King Charles II of England granted the lands to his brother, the Duke of York, who named the city New York in his own honor.",
    "The Statue of Liberty in New York City holds great significance as a symbol of the United States and its ideals of liberty and peace. It greeted millions of immigrants who arrived in the U.S. by ship in the late 19th and early 20th centuries, representing hope and freedom for those seeking a better life. It has since become an iconic landmark and a global symbol of cultural diversity and freedom.",
]

examples = [
    {"query": q, "ground_truths": [eval_answers[i]]}
    for i, q in enumerate(eval_questions)
]

In [29]:
# Generate predictions with your qa_chain
all_predictions = []
for example in examples:
    q = example["query"]
    result = qa_chain.invoke({"query": q})  # RetrievalQA uses 'query'

    all_predictions.append({
        "question": q,
        "answer": result['result'],  # Extract answer from result dict
        "contexts": [doc.page_content for doc in result['source_documents']],  # Extract contexts
        "ground_truth": example["ground_truths"][0]
    })

print("Predictions generated:", len(all_predictions))

# Display first prediction to verify
print("\nExample prediction:")
print("Question:", all_predictions[0]['question'])
print("Answer:", all_predictions[0]['answer'])
print("Contexts:", len(all_predictions[0]['contexts']), "documents")
print("Ground Truth:", all_predictions[0]['ground_truth'])

Predictions generated: 5

Example prediction:
Question: What is the population of New York City as of 2020?
Answer: The population of New York City as of 2020 is 8,804,190 residents.
Contexts: 4 documents
Ground Truth: 8,804,190


#### Introducing RagasEvaluatorChain

`RagasEvaluatorChain` creates a wrapper around the metrics ragas provides (documented [here](https://github.com/explodinggradients/ragas/blob/main/docs/metrics.md)), making it easier to run these evaluation with langchain and langsmith.

The evaluator chain has the following APIs

- `__call__()`: call the `RagasEvaluatorChain` directly on the result of a QA chain.
- `evaluate()`: evaluate on a list of examples (with the input queries) and predictions (outputs from the QA chain).
- `evaluate_run()`: method implemented that is called by langsmith evaluators to evaluate langsmith datasets.

lets see each of them in action to learn more.

evaluate(): evaluate on a list of examples (with the input queries) and predictions (outputs from the QA chain).

In [30]:
# FIX: nest_asyncio prevents 'asyncio.run() cannot be called from a running event loop'
import nest_asyncio
nest_asyncio.apply()

import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

from ragas import evaluate
from ragas.metrics import faithfulness, context_recall
from datasets import Dataset

# all_predictions already has 'contexts' as list of strings (built above)
dataset_all = Dataset.from_list(all_predictions)

print("=" * 50)
print("RAGAS Evaluation - Faithfulness")
print("=" * 50)
f = evaluate(dataset_all, metrics=[faithfulness])
print(f)

print("\n" + "=" * 50)
print("RAGAS Evaluation - Context Recall")
print("=" * 50)
r = evaluate(dataset_all, metrics=[context_recall])
print(r)

print("\n" + "=" * 50)
print("RAGAS Evaluation - Combined Metrics")
print("=" * 50)
results = evaluate(dataset_all, metrics=[faithfulness, context_recall])
print(results)


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_3094/1391420488.py:10: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, context_recall
/tmp/ipykernel_3094/1391420488.py:10: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  

RAGAS Evaluation - Faithfulness


Evaluating:   0%|          | 0/5 [00:00<?, ?it/s]

{'faithfulness': 0.9714}

RAGAS Evaluation - Context Recall


Evaluating:   0%|          | 0/5 [00:00<?, ?it/s]

{'context_recall': 0.7333}

RAGAS Evaluation - Combined Metrics


Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]

{'faithfulness': 0.9429, 'context_recall': 0.7333}


In [31]:
# FIX: build result_updated with the correct RAGAS keys and string contexts
# Re-run the chain so 'result' is in scope for downstream cells
result = qa_chain.invoke({'query': eval_questions[1]})

result_updated = {
    'question': result['query'],
    'answer':   result['result'],
    'contexts': [doc.page_content for doc in result['source_documents']],  # strings, not Documents
    'ground_truth': eval_answers[1],  # required for context_recall
}
print('Answer:', result_updated['answer'])


Answer: Manhattan (New York County) has the highest population density of any borough in New York City.


In [32]:
print("=" * 70)
print("RAGAS EVALUATION WITH EvaluatorChain")
print("=" * 70)

import nest_asyncio
nest_asyncio.apply()  # FIX: required for Jupyter async compatibility

from ragas.integrations.langchain import EvaluatorChain
from ragas.metrics import faithfulness, context_recall

faithfulness_chain = EvaluatorChain(metric=faithfulness)
context_recall_chain = EvaluatorChain(metric=context_recall)

print("✓ Evaluation chains created")


RAGAS EVALUATION WITH EvaluatorChain


/usr/local/lib/python3.12/dist-packages/pydantic/v1/main.py:1002: RuntimeWarning: fields may not start with an underscore, ignoring "_required_columns"
  warnings.warn(f'fields may not start with an underscore, ignoring "{f_name}"', RuntimeWarning)
/tmp/ipykernel_3094/2671089951.py:9: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, context_recall
/tmp/ipykernel_3094/2671089951.py:9: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import faithfulness, context_recall


✓ Evaluation chains created


Method 1: Evaluate a Single Result with __call__()

In [33]:
# Method 1: Evaluate a Single Result with __call__()

print("\n" + "=" * 70)
print("METHOD 1: Evaluate Single Result with __call__()")
print("=" * 70)

single_query = examples[0]["query"]
print(f"\nQuery: {single_query}")

result = qa_chain.invoke({"query": single_query})
print(f"Answer: {result['result']}")

single_eval_input = {
    "question":    result["query"],
    "answer":      result["result"],
    "contexts":    [doc.page_content for doc in result["source_documents"]],  # FIX: strings
    "ground_truth": examples[0]["ground_truths"][0],
}

print("\n--- Evaluating Faithfulness ---")
faithfulness_result = faithfulness_chain(single_eval_input)
print(f"Faithfulness Score: {faithfulness_result.get('faithfulness', 'N/A')}")
  # FIX: key is 'faithfulness', not 'faithfulness_score'

print("\n--- Evaluating Context Recall ---")
context_recall_result = context_recall_chain(single_eval_input)
print(f"Context Recall Score: {context_recall_result.get('context_recall', 'N/A')}")
  # FIX: key is 'context_recall', not 'context_recall_score'



METHOD 1: Evaluate Single Result with __call__()

Query: What is the population of New York City as of 2020?
Answer: The population of New York City as of 2020 is 8,804,190 residents.

--- Evaluating Faithfulness ---


/tmp/ipykernel_3094/2587188287.py:21: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use invoke instead.
  faithfulness_result = faithfulness_chain(single_eval_input)


Faithfulness Score: 0.5

--- Evaluating Context Recall ---
Context Recall Score: 1.0


Test with Fake Results (Low Scores)

In [34]:
# Test with Fake Results (Low Scores)

print("\n" + "=" * 70)
print("TESTING WITH FAKE RESULTS (Expected Low Scores)")
print("=" * 70)

# FIX: build base dict fresh with string contexts (not Documents)
base_eval_input_for_test = {
    "question":    result["query"],
    "answer":      result["result"],
    "contexts":    [doc.page_content for doc in result["source_documents"]],
    "ground_truth": examples[0]["ground_truths"][0],
}

# Test 1: Fake answer → should give low faithfulness
print("\n--- Test 1: Fake Answer ---")
fake_faithfulness_input = base_eval_input_for_test.copy()
fake_faithfulness_input["answer"] = "The population of NYC is 100 million and it's on Mars."
fake_faithfulness = faithfulness_chain(fake_faithfulness_input)
# FIX: use .get() with fallback in case LLM extracts no statements
orig_f = faithfulness_result.get('faithfulness', 'N/A')
fake_f = fake_faithfulness.get('faithfulness', 0.0)
print(f"Original Faithfulness: {orig_f}")
print(f"Fake Faithfulness:     {fake_f}")
print("→ Lower score because answer contradicts the source documents")

# Test 2: Fake contexts → should give low context recall
print("\n--- Test 2: Fake Source Documents ---")
fake_context_recall_input = base_eval_input_for_test.copy()
# FIX: pass strings directly — no Document objects needed here
fake_context_recall_input["contexts"] = [
    "I love pizza and ice cream.",
    "The weather is nice today."
]
fake_context_recall = context_recall_chain(fake_context_recall_input)
orig_r = context_recall_result.get('context_recall', 'N/A')
fake_r = fake_context_recall.get('context_recall', 0.0)
print(f"Original Context Recall: {orig_r}")
print(f"Fake Context Recall:     {fake_r}")
print("→ Lower score because ground truth is not in source documents")



TESTING WITH FAKE RESULTS (Expected Low Scores)

--- Test 1: Fake Answer ---
Original Faithfulness: 0.5
Fake Faithfulness:     0.0
→ Lower score because answer contradicts the source documents

--- Test 2: Fake Source Documents ---
Original Context Recall: 1.0
Fake Context Recall:     0.0
→ Lower score because ground truth is not in source documents


Method 2: Batch Evaluate with evaluate()

In [35]:
#Method 2: Batch Evaluate with evaluate()

def evaluate_qa_pipeline(
    qa_chain,
    examples,
    evaluators,
    prediction_key="result",
    context_key="source_documents",
):
    print("\n" + "=" * 70)
    print("RUNNING BATCH EVALUATION")
    print("=" * 70)

    # Generate predictions
    predictions = qa_chain.batch([{"query": ex["query"]} for ex in examples])

    def extract_contexts(pred):
        docs = pred.get(context_key, [])
        return [doc.page_content for doc in docs] if docs else []

    results = {"metrics": {}}

    for name, evaluator in evaluators.items():
        print(f"\n--- Evaluating {name} ---")

        eval_inputs = [
            {
                "question": ex["query"],
                "answer": pred.get(prediction_key, ""),
                "contexts": extract_contexts(pred),
                "ground_truth": ex["ground_truths"][0] if ex.get("ground_truths") else "",
            }
            for ex, pred in zip(examples, predictions)
        ]

        eval_outputs = evaluator.batch(eval_inputs)

        # Flexible score extraction
        def extract_score(res):
            if "score" in res:
                return res["score"]
            if name in res:
                return res[name]
            if "value" in res:
                return res["value"]
            return None

        scores = []
        for res in eval_outputs:
            score = extract_score(res)
            if score is not None:
                scores.append(score)
            else:
                print("Unknown format:", res)

        avg_score = sum(scores) / len(scores) if scores else 0.0

        print(f"Average {name}: {avg_score:.3f}")
        print(f"Scores: {scores}")

        results["metrics"][name] = {
            "average": avg_score,
            "scores": scores,
        }

    return results

In [36]:
evaluators = {
    "faithfulness": faithfulness_chain,
    "context_recall": context_recall_chain,
}

results = evaluate_qa_pipeline(
    qa_chain=qa_chain,
    examples=examples,
    evaluators=evaluators,
)


RUNNING BATCH EVALUATION

--- Evaluating faithfulness ---
Average faithfulness: 0.900
Scores: [1.0, 0.5, 1.0, 1.0, 1.0]

--- Evaluating context_recall ---
Average context_recall: 0.833
Scores: [1.0, 1.0, 1.0, 0.5, 0.6666666666666666]


In [37]:
# Summary of batch evaluation results
print("\n" + "=" * 70)
print("BATCH EVALUATION SUMMARY")
print("=" * 70)
for metric, data in results['metrics'].items():
    print(f"{metric:20s}  avg={data['average']:.3f}  scores={[round(s,3) for s in data['scores']]}")



BATCH EVALUATION SUMMARY
faithfulness          avg=0.900  scores=[1.0, 0.5, 1.0, 1.0, 1.0]
context_recall        avg=0.833  scores=[1.0, 1.0, 1.0, 0.5, 0.667]


SECTION 1 — Single Example Evaluation (Understanding RAGAS Basics)

In [38]:
# SECTION 1 — Single Example Evaluation (Understanding RAGAS Basics)

import nest_asyncio
nest_asyncio.apply()  # FIX: Jupyter async compatibility

from ragas.integrations.langchain import EvaluatorChain
from ragas.metrics import faithfulness, context_recall

faithfulness_chain = EvaluatorChain(metric=faithfulness)
context_recall_chain = EvaluatorChain(metric=context_recall)

query = examples[0]["query"]
result = qa_chain.invoke({"query": query})

single_eval_input = {
    "question":    query,
    "answer":      result["result"],
    "contexts":    [doc.page_content for doc in result["source_documents"]],  # FIX: strings
    "ground_truth": examples[0]["ground_truths"][0],
}

faith_score = faithfulness_chain(single_eval_input)
recall_score = context_recall_chain(single_eval_input)

# FIX: correct key names — 'faithfulness' and 'context_recall' (no _score suffix)
print("Faithfulness:",   faith_score.get('faithfulness', 'N/A'))
print("Context Recall:", recall_score.get('context_recall', 'N/A'))


/tmp/ipykernel_3094/2987150844.py:7: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, context_recall
/tmp/ipykernel_3094/2987150844.py:7: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import faithfulness, context_recall


Faithfulness: 0.5
Context Recall: 1.0


SECTION 2 — Batch Evaluation (LangChain Style)

In [39]:
def run_batch_evaluation(qa_chain, examples):
    predictions = []

    for ex in examples:
        result = qa_chain.invoke({"query": ex["query"]})

        predictions.append({
            "question": ex["query"],
            "answer": result["result"],
            "contexts": [doc.page_content for doc in result["source_documents"]],
            "ground_truth": ex["ground_truths"][0],
        })

    return predictions


predictions = run_batch_evaluation(qa_chain, examples)

In [40]:
eval_inputs = [
    {
        "question": p["question"],
        "answer": p["answer"],
        "contexts": p["contexts"],
        "ground_truth": p["ground_truth"],
    }
    for p in predictions
]

faith_batch = faithfulness_chain.batch(eval_inputs)
recall_batch = context_recall_chain.batch(eval_inputs)

# average manually
faith_scores = [r["faithfulness"] for r in faith_batch]
recall_scores = [r["context_recall"] for r in recall_batch]

print("Avg Faithfulness:", sum(faith_scores)/len(faith_scores))
print("Avg Context Recall:", sum(recall_scores)/len(recall_scores))

Avg Faithfulness: 0.9
Avg Context Recall: 0.9333333333333333


SECTION 3 — RAGAS Native Evaluation (Production Method)

In [42]:
from ragas import evaluate
from datasets import Dataset

dataset_all = Dataset.from_list([
    {
        "question":  ex["query"],
        "answer":    pred["answer"],
        "contexts":  pred["contexts"],
        "reference": ex["ground_truths"][0],  # FIX: renamed + unwrap list to string
    }
    for ex, pred in zip(examples, predictions)
])

results = evaluate(
    dataset_all,
    metrics=[faithfulness, context_recall]
)
print(results)

Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]

{'faithfulness': 0.9000, 'context_recall': 0.7333}
